# Chapter 5 -- TF-IDF：把文字变成数字

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chaosfrey-arch/news-classification-tutorial/blob/main/chapter05_tfidf.ipynb)

**本章目标：** 理解 TF-IDF 的原理，并用 sklearn 实现。

**预计时间：** 45 分钟

> 参考：Karen Sparck Jones 1972 年提出 IDF 概念，奠定了现代信息检索的基础。

> 参考：[Karen Sparck Jones 原始论文（1972）](https://citeseerx.ist.psu.edu/viewdoc/summary?doi=10.1.1.115.8343) | [sklearn TfidfVectorizer 文档](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) | [TF-IDF 图解（Towards Data Science）](https://towardsdatascience.com/tf-idf-simplified-aba19d5f5530)

## 5.1 机器只懂数字

你现在明白：模型需要输入数据、输出预测。

但问题是：**文章是文字，模型只接受数字。**

所以我们必须把文章先转换成数字。这个步骤叫**文本表示（Text Representation）**。

## 5.2 最简单的方法：词袋模型（Bag of Words）

**词袋模型**：不管词的顺序，只统计每个词出现了几次。

类比：把文章里所有词「倒进一个袋子」，摇一摇，然后数每种词有多少个。

**手算例子：**

```
文章 1："cat dog cat"
文章 2："dog fish"
文章 3："cat fish fish"

词汇表：[cat, dog, fish]

          cat  dog  fish
文章 1  [  2,   1,   0 ]
文章 2  [  0,   1,   1 ]
文章 3  [  1,   0,   2 ]
```

每篇文章变成了一行数字！这就是词袋矩阵。

## 5.3 词袋模型的问题

"the"、"is"、"and" 这类常见词在所有文章里都大量出现，数量高，但对判断类别没有帮助。

我们需要一种方法，让「独特的词」权重高，让「到处都有的词」权重低。

这就是 **TF-IDF**。

## 5.4 TF：词频（Term Frequency）

**TF(词, 文章)** = 这个词在这篇文章里出现了多少次（相对频率）

```
TF("quarterback", 文章A) = 文章A中"quarterback"出现次数 / 文章A总词数
```

但问题还没解决："the" 的 TF 也很高，因为它到处都出现。

## 5.5 IDF：逆文档频率（Inverse Document Frequency）

**IDF(词)** = 衡量这个词有多稀有

```
IDF(词) = log( 总文档数 / 包含该词的文档数 )
```

直觉解释：
- "the" 在 128,000 篇文章里都出现 → IDF 接近 0（没价值）
- "quarterback" 只在 500 篇体育文章里出现 → IDF 很高（很有价值）

| 词 | 出现在多少文章里 | IDF（近似）|
|---|---|---|
| the | 128,000 | ≈ 0（没区分度）|
| sports | 35,000 | 低 |
| quarterback | 500 | 高 |
| photosynthesis | 50 | 很高 |

## 5.6 TF-IDF = TF × IDF

$$\text{TF-IDF}(词, 文章) = \text{TF}(词, 文章) \times \text{IDF}(词)$$

- 一个词在这篇文章里多 → TF 高
- 这个词在所有文章里少见 → IDF 高
- 两者乘积高的词，就是最能代表这篇文章的词

**举例：** 一篇体育新闻里频繁出现 "quarterback"（TF 高），而 "quarterback" 只出现在体育文章里（IDF 高）→ TF-IDF 高 → 这个词非常能代表这篇文章是体育新闻。

In [ ]:
# 手动验证 TF-IDF 的直觉
import numpy as np

# 3 篇迷你文章
docs = [
    "the cat sat on the mat",
    "the dog played football at the stadium",
    "the scientist discovered a new chemical compound"
]

# 用 sklearn 计算 TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(docs)

# 特征名（词汇表）
feature_names = vectorizer.get_feature_names_out()

print('词汇表：', feature_names.tolist())
print('\nTF-IDF 矩阵（每行是一篇文章，每列是一个词）：')
print(np.round(X.toarray(), 3))

# 找每篇文章权重最高的词
print('\n每篇文章权重最高的词：')
for i, doc in enumerate(docs):
    tfidf_scores = X[i].toarray()[0]
    top_idx = tfidf_scores.argsort()[-3:][::-1]
    print(f'  文章{i+1}: {[feature_names[j] for j in top_idx]}')

## 5.7 在 dataset.csv 上实现 TF-IDF

In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# ── 数据准备（和 Chapter 3 相同）──────────────────────────
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df = pd.read_csv('dataset.csv').dropna(subset=['Class']).copy()
df['Description'] = df['Description'].fillna('')
df['text'] = (df['Title'] + ' ' + df['Description']).apply(clean_text)
le = LabelEncoder()
y = le.fit_transform(df['Class'])
X_train, X_test, y_train, y_test = train_test_split(
    df['text'].values, y, test_size=0.3, random_state=42, stratify=y)
print(f'训练集：{len(X_train):,}，测试集：{len(X_test):,}')

In [ ]:
# 创建 TF-IDF 向量化器
tfidf = TfidfVectorizer(
    max_features=20000,    # 只保留最重要的 20000 个词（词汇表大小）
    ngram_range=(1, 2),    # 考虑单词和两词组合（bigram）
                           # 例如 'stock market' 比 'stock' 和 'market' 单独更有价值
    sublinear_tf=True,     # 用 1+log(tf) 代替原始 tf，防止高频词垄断
    min_df=2               # 至少在 2 篇文章里出现，过滤极少见的词
)

# fit_transform：在训练集上学习词汇表，并转换训练集
# （重要！只在训练集上 fit，测试集只 transform）
X_train_tfidf = tfidf.fit_transform(X_train)

# transform：只用学到的词汇表转换测试集（不重新学习）
X_test_tfidf = tfidf.transform(X_test)

print(f'训练集 TF-IDF 矩阵形状：{X_train_tfidf.shape}')
print(f'  → {X_train_tfidf.shape[0]:,} 篇文章，每篇用 {X_train_tfidf.shape[1]:,} 个数字表示')
print(f'\n稀疏度：{(1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0]*X_train_tfidf.shape[1])):.1%}')
print('（绝大多数位置是 0，因为每篇文章只用到极少数词）')

## 5.8 为什么只在训练集上 fit？

**答案：防止数据泄露（Data Leakage）**

如果在全部数据（包括测试集）上 fit TF-IDF：
- 词汇表里会包含测试集独有的词
- 这意味着模型「偷看」了测试集
- 评估结果就不真实了

正确做法：
```python
tfidf.fit_transform(X_train)  # 在训练集上学词汇表并转换
tfidf.transform(X_test)       # 用同一个词汇表转换测试集
```

In [ ]:
# 查看某篇文章权重最高的 10 个词
import numpy as np

# 取第一篇训练文章
feature_names = tfidf.get_feature_names_out()
sample = X_train_tfidf[0]

# 找权重最高的词
top_indices = sample.toarray()[0].argsort()[-10:][::-1]
print('第一篇训练文章的 Top 10 TF-IDF 词：')
for idx in top_indices:
    print(f'  {feature_names[idx]:30s} {sample.toarray()[0][idx]:.4f}')

print('\n对应的文章类别：', le.classes_[y_train[0]])

## 练习

In [ ]:
# 练习：找出「Sports」类文章中 TF-IDF 权重最高的 20 个词
# 提示：先用 y_train==2（Sports 的编码）筛选出运动类文章，
# 然后对这些文章的 TF-IDF 矩阵求平均

import numpy as np

# 你的代码：
# sports_mask = (y_train == ?)  # Sports 是哪个编码？用 le.transform(['Sports'])
# sports_tfidf = X_train_tfidf[sports_mask]
# avg_tfidf = np.array(sports_tfidf.mean(axis=0)).flatten()
# top20 = avg_tfidf.argsort()[-20:][::-1]
# for i in top20:
#     print(feature_names[i], avg_tfidf[i])


## 总结

| 概念 | 含义 |
|------|------|
| 词袋模型（BoW）| 只数词频，不管顺序 |
| TF（词频）| 词在本文中出现的频率 |
| IDF（逆文档频率）| 词在所有文章中有多稀有 |
| TF-IDF | TF × IDF，突出独特词 |
| max_features | 词汇表大小上限 |
| ngram_range=(1,2) | 同时考虑单词和双词组合 |
| fit 只用训练集 | 防止数据泄露 |

**下一章 →** Chapter 6：第一个分类器——朴素贝叶斯